# 第2回：画像認識の基礎（カメラ・画像・動画からデータを取る）

**このNotebookでできるようになること**

1. 画像ファイル・動画ファイル・Webカメラの映像を読み込んで表示する
2. 動画をフレーム（静止画）に分割して保存する
3. 画像・映像の中から **人物** と **顔** を検出する
4. 顔の向きを推定する（dlib / MediaPipe）
5. 検出結果をまとめてタイムラプス動画を作る

> **使い方**：上から順にセルを実行（`Shift + Enter`）して，結果を確認してください．
> Webカメラを使うセルは別ウィンドウが開きます．**ウィンドウを選択した状態で `q` キー**を押すと終了します．

## 事前準備

`image_recognition/README.md` の手順で `download_assets.py` を実行し，次のファイルを用意してください．

```
image_recognition/
├── data/
│   ├── img/img01.jpg, img02.jpg     # 人物が写った画像（自分で用意した写真でもよい）
│   ├── vtest.avi                    # 歩行者の動画（OpenCV公式サンプル）
│   └── shape_predictor_68_face_landmarks.dat   # dlib の顔ランドマークモデル
└── det/
    ├── coco.names
    └── yolov5s.onnx                 # 第4回で使用
```

## 使うライブラリ

| ライブラリ | 役割 |
|---|---|
| **OpenCV** (`cv2`) | 画像・動画の読み書き，色変換，古典的な検出器（HOG，Haar） |
| **dlib** | 顔検出と68点の顔ランドマーク |
| **MediaPipe** | Googleの軽量な深層学習モデル（顔メッシュ，姿勢，手） |
| **matplotlib** | Notebook上での画像・グラフ表示 |

## 0. 準備

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV:", cv2.__version__)

def show(img_bgr, title="", size=(8, 6)):
    # OpenCVの画像(BGR)をNotebook上に表示する小さな関数（RGBに変換してから表示）
    plt.figure(figsize=size)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

## 1. 画像ファイルを読み込む

OpenCV は画像を **NumPy配列（高さ × 幅 × 3色）** として扱います．色の並びが RGB ではなく **BGR** なので，matplotlib で表示する前に変換します．

In [ ]:
img = cv2.imread("data/img/img01.jpg")
if img is None:
    raise FileNotFoundError("data/img/img01.jpg がありません．download_assets.py を実行するか，自分の写真を置いてください．")

height, width = img.shape[:2]
print("画像幅:", width, " 画像高さ:", height, " 配列の形:", img.shape, " 型:", img.dtype)
show(img, "img01.jpg")

In [ ]:
# 画素の値を見る：左上 (y=0, x=0) の画素の B, G, R
print("左上の画素 (B,G,R):", img[0, 0])

# グレースケール（白黒）に変換して表示
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
plt.figure(figsize=(8, 6))
plt.imshow(gray, cmap="gray")
plt.title("グレースケール")
plt.axis("off")
plt.show()

## 2. 動画ファイルを読み込む

動画は「フレーム（静止画）の連続」です．`cv2.VideoCapture` で開き，`read()` で1フレームずつ取り出します．

このセルは別ウィンドウに動画を再生します．ウィンドウを選んで `q` を押すと終了します．

In [ ]:
cap = cv2.VideoCapture("data/vtest.avi")

width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"画像幅: {width}  画像高さ: {height}  総フレーム数: {count}  FPS: {fps:.2f}")

while cap.isOpened():
    ret, frame = cap.read()          # ret: 読めたか  frame: 画像
    if not ret:
        break                        # 動画の最後
    resized = cv2.resize(frame, (int(width // 2), int(height // 2)))
    cv2.imshow("frame", resized)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## 3. Webカメラの映像を読み込む

`cv2.VideoCapture(0)` で内蔵カメラを開きます（外付けなら 1, 2, …）．Windows では `cv2.CAP_DSHOW` を付けると起動が速くなります（Macでは外してください）．

In [ ]:
# [SKIP-TEST]
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)     # Mac の場合: cv2.VideoCapture(0)
if not cap.isOpened():
    print("カメラを開けません．カメラの番号やアクセス許可を確認してください．")

print("画像幅:", cap.get(cv2.CAP_PROP_FRAME_WIDTH), " 画像高さ:", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    cv2.imshow("Webcam", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## 4. 動画をフレームに分割して保存する

深層学習の学習データを作るときの基本操作です．全フレームを保存すると枚数が膨大になるので，ここでは **30フレームに1枚** 保存します．

In [ ]:
os.makedirs("data/snapshot", exist_ok=True)
cap = cv2.VideoCapture("data/vtest.avi")
num = saved = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    if num % 30 == 0:
        cv2.imwrite(f"data/snapshot/snapshot_{num:05d}.jpg", frame)
        saved += 1
    num += 1
cap.release()
print(f"{num} フレーム中 {saved} 枚を data/snapshot/ に保存しました")
show(cv2.imread("data/snapshot/snapshot_00000.jpg"), "保存した1枚目")

## 5. 画像の中の人物を検出する（HOG + SVM）

深層学習以前から使われている古典的な人物検出器です．
**HOG**（輝度の勾配方向のヒストグラム）という特徴を計算し，**SVM** で「人物らしいか」を判定します．

In [ ]:
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())
hogParams = {"winStride": (8, 8), "padding": (32, 32), "scale": 1.05, "hitThreshold": 0}

img = cv2.imread("data/img/img01.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
human, weights = hog.detectMultiScale(gray, **hogParams)   # 検出された矩形 (x, y, w, h) のリスト
print("検出された人物の数:", len(human))

for (x, y, w, h) in human:
    cv2.rectangle(img, (x, y), (x + w, y + h), (255, 255, 255), 3)
show(img, f"HOG人物検出: {len(human)} 人")

## 6. 画像の中の顔を検出する（Haar Cascade）

顔検出の古典的手法．学習済みの分類器（XML）は OpenCV に同梱されているので，ダウンロード不要です．

In [ ]:
cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_alt.xml")

img = cv2.imread("data/img/img02.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
faces = cascade.detectMultiScale(gray, minSize=(50, 50))
print("検出された顔の数:", len(faces))

for (x, y, w, h) in faces:
    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 0, 225), 3)
show(img, f"Haar顔検出: {len(faces)} 人")

## 7. 顔の向きを推定する（dlib の68点ランドマーク）

dlib は顔の輪郭・目・鼻・口の **68個の特徴点** を検出できます．
輪郭の重心と内側（目鼻口）の重心のずれから，顔が左右どちらを向いているかを推定します．

> dlib のインストールに失敗した場合，この節と次の節は飛ばして，8節の MediaPipe を使ってください．

In [ ]:
import math
import dlib

predictor = dlib.shape_predictor("data/shape_predictor_68_face_landmarks.dat")
detector = dlib.get_frontal_face_detector()

img = cv2.imread("data/img/img02.jpg")
dets = detector(img, 1)                       # 1 はアップサンプリング回数
print("検出された顔の数:", len(dets))

for k, d in enumerate(dets):
    shape = predictor(img, d)                 # 68点のランドマーク
    cv2.rectangle(img, (d.left(), d.top()), (d.right(), d.bottom()), (0, 0, 225), 3)

    n_out = 17                                # 0〜16 が輪郭，17〜67 が目鼻口
    n_in = shape.num_parts - n_out
    gx_out = gy_out = gx_in = gy_in = 0
    for i in range(shape.num_parts):
        p = shape.part(i)
        if i < n_out:
            cv2.circle(img, (p.x, p.y), 3, (255, 0, 0), 3)
            gx_out += p.x / n_out; gy_out += p.y / n_out
        else:
            cv2.circle(img, (p.x, p.y), 3, (0, 255, 0), 3)
            gx_in += p.x / n_in; gy_in += p.y / n_in

    theta = math.asin(max(-1, min(1, 2 * (gx_in - gx_out) / (d.right() - d.left()))))
    deg = theta * 180 / math.pi
    label = f"{'left' if deg < 0 else 'right'} {abs(deg):.1f} deg"
    print(f"顔{k}: {label}")
    cv2.putText(img, label, (d.left(), d.top() - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 225), 3)

show(img, "dlib 顔ランドマークと向き")

### Webカメラでリアルタイムに顔の向きを推定する

In [ ]:
# [SKIP-TEST]
import math
import dlib

predictor = dlib.shape_predictor("data/shape_predictor_68_face_landmarks.dat")
detector = dlib.get_frontal_face_detector()
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    for k, d in enumerate(detector(gray, 1)):
        shape = predictor(frame, d)
        cv2.rectangle(frame, (d.left(), d.top()), (d.right(), d.bottom()), (0, 0, 225), 2)
        n_out = 17; n_in = shape.num_parts - n_out
        gx_out = gx_in = 0
        for i in range(shape.num_parts):
            p = shape.part(i)
            cv2.circle(frame, (p.x, p.y), 2, (255, 0, 0) if i < n_out else (0, 255, 0), 2)
            if i < n_out: gx_out += p.x / n_out
            else: gx_in += p.x / n_in
        theta = math.asin(max(-1, min(1, 2 * (gx_in - gx_out) / (d.right() - d.left()))))
        deg = theta * 180 / math.pi
        cv2.putText(frame, f"{'left' if deg < 0 else 'right'} {abs(deg):.1f} deg", (d.left(), d.top() - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 225), 2)
    cv2.imshow("Webcam", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## 8. MediaPipe Face Mesh で顔の向きを推定する（深層学習版）

MediaPipe は Google が公開している軽量な深層学習モデル群です．Face Mesh は **468点** の顔メッシュを高速に検出します．
鼻先が顔の中心からどれだけずれているかで左右の向きを推定します．

In [ ]:
# [SKIP-TEST]
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(max_num_faces=5, refine_landmarks=True,
                                  min_detection_confidence=0.5, min_tracking_confidence=0.5)

NOSE_TIP, LEFT_CHEEK, RIGHT_CHEEK, FOREHEAD, CHIN = 1, 234, 454, 10, 152

def face_yaw(lm, w):
    # 鼻先の x が，左右の頬の中点からどれだけずれているかを角度（±45度程度）に換算
    cx = (lm[LEFT_CHEEK].x + lm[RIGHT_CHEEK].x) / 2 * w
    fw = abs(lm[RIGHT_CHEEK].x - lm[LEFT_CHEEK].x) * w
    dev = (lm[NOSE_TIP].x * w - cx) / max(fw, 1) * 2
    return max(-0.8, min(0.8, dev)) * 45

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    h, w = frame.shape[:2]
    results = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if results.multi_face_landmarks:
        for face in results.multi_face_landmarks:
            lm = face.landmark
            xs = [lm[i].x * w for i in (LEFT_CHEEK, RIGHT_CHEEK, FOREHEAD, CHIN)]
            ys = [lm[i].y * h for i in (LEFT_CHEEK, RIGHT_CHEEK, FOREHEAD, CHIN)]
            x0, x1, y0, y1 = int(min(xs)), int(max(xs)), int(min(ys)), int(max(ys))
            angle = face_yaw(lm, w)
            direction = "LEFT" if angle < -5 else "RIGHT" if angle > 5 else "CENTER"
            cv2.rectangle(frame, (x0, y0), (x1, y1), (0, 0, 225), 2)
            cv2.putText(frame, f"{direction} {abs(angle):.1f} deg", (x0, max(y0 - 10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.circle(frame, (int(lm[NOSE_TIP].x * w), int(lm[NOSE_TIP].y * h)), 4, (0, 0, 255), -1)
    cv2.imshow("MediaPipe Face Direction", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

face_mesh.close()
cap.release()
cv2.destroyAllWindows()

## 9. 検出結果を統合してタイムラプス動画を作る

10フレームごとに人物検出を行い，結果を描いたフレームを新しい動画に書き出します．
「長時間の映像を短時間で確認する」「検出結果を記録する」といった応用の基本形です．

In [ ]:
cap = cv2.VideoCapture("data/vtest.avi")
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())
hogParams = {"winStride": (8, 8), "padding": (32, 32), "scale": 1.05, "hitThreshold": 0}

fourcc = cv2.VideoWriter_fourcc(*"XVID")
video = cv2.VideoWriter("data/timelapse.avi", fourcc, 30, (width, height))

num = 0
print("タイムラプス生成を開始します")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    if num % 10 == 0:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        human, _ = hog.detectMultiScale(gray, **hogParams)
        for (x, y, w, h) in human:
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 255), 3)
        video.write(frame)
    num += 1

video.release()
cap.release()
print("data/timelapse.avi を書き出しました")

## まとめ

| できること | 使ったもの |
|---|---|
| 画像・動画・カメラの読み込みと保存 | `cv2.imread`, `cv2.VideoCapture`, `cv2.imwrite`, `cv2.VideoWriter` |
| 人物検出 | HOG + SVM（`cv2.HOGDescriptor`） |
| 顔検出 | Haar Cascade（`cv2.CascadeClassifier`） |
| 顔の向き | dlib 68点ランドマーク / MediaPipe Face Mesh |

**次回（第3回）**：検出結果を **数値データ** として記録し，グラフ化・変化点検出などの分析を行います．